
# RNN vs LSTM vs GRU

## Instructions

In this lab, you will build **three text classification models** from scratch:
- RNN
- LSTM
- GRU

---

### Objectives
By the end of this lab, you should be able to:

- Preprocess text data
- Build a vocabulary
- Encode and pad sequences
- Implement RNN, LSTM, and GRU in PyTorch
- Train and evaluate models 
- Compare architectures


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from datasets import load_dataset

from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [4]:
# data loading

dataset = load_dataset("ag_news")

# Reduce dataset size for faster training during class
train_data = list(dataset["train"])[:5000]
test_data = list(dataset["test"])[:1000]

print("Train size:", len(train_data))
print("Test size:", len(test_data))


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Train size: 5000
Test size: 1000



# Part 1 – Text Preprocessing

You must:

1. Write a `tokenize(text)` function.
2. Build a vocabulary using the training set only.
3. Keep only the top 10,000 most frequent words.
4. Add special tokens:
   - `<pad>`
   - `<unk>`
5. Explain in a markdown cell:
   - Why do we not build the vocabulary using the test set?


In [ ]:
import re
def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.split()

def build_vocab(texts, max_words=10000):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
        
    most_common = counter.most_common(max_words)
    vocab = {"<pad>": 0, "<unk>": 1}
    for i, (word, _) in enumerate(most_common, start=2):
        vocab[word] = i
        
    return vocab

# The texts for the vocabulary build comes ONLY from the training dataset
train_texts_collection = [example['text'] for example in train_data]
vocab = build_vocab(train_texts_collection, max_words=10000)

print(f"Vocabulary built. Size: {len(vocab)}")


### Explanation

- **Why do we not build the vocabulary using the test set?**
Building a vocabulary using the test set inadvertently exposes the model to terms and characteristics of the evaluation unseen data (often called "Data Leakage"). Specifically, the presence and counts/frequencies of words inside the test set should be an absolute black box prior to evaluation to remain robust and truly test how the model behaves upon out-of-distribution or entirely novel inputs.



# Part 2 – Encoding and Padding

You must:

1. Create an `encode(text)` function.
2. Convert tokens into vocabulary indices.
3. Pad or truncate sequences to a fixed length (e.g., 25).
4. Create a custom `collate()` function.
5. Create train, validation, and test DataLoaders.

Explain:
- Why is padding necessary?
- Why should validation and test loaders not shuffle?


In [ ]:
def encode(vocab, text, max_len=25):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    # Truncate
    ids = ids[:max_len]
    # Pad
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
        
    return torch.tensor(ids, dtype=torch.long)

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, data, vocab, max_len=25):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        x = encode(self.vocab, item['text'], self.max_len)
        y = torch.tensor(item['label'], dtype=torch.long) # CrossEntropyLoss wants long/indices
        return x, y

# Validation data split from train_data
val_size = int(0.2 * len(train_data))
train_size = len(train_data) - val_size

train_subset, val_subset = random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = NewsDataset(train_subset, vocab, max_len=30)
val_dataset = NewsDataset(val_subset, vocab, max_len=30)
test_dataset = NewsDataset(test_data, vocab, max_len=30)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


### Explanation

- **Why is padding necessary?**
  Neural networks train efficiently utilizing tensor batches mapping highly vectorized operations on CPUs/GPUs. Operations on tensors require arrays to have homogenous/fixed topological dimensions. By padding, all varying sequence lengths match to precisely one shared static maximum length `max_len`. 
- **Why should validation and test loaders not shuffle?**
  Shuffling disrupts the deterministic structure of evaluating elements. Without shuffling sequentially, we have reliable and fixed metric evaluations, enabling deterministic analysis, correct confusion matrix matching index to labels, and keeping specific sequences in order for inspection or sequential decoding if desired.



# Part 3 – Model Implementation

Create a class called `Model` that:

- Uses an Embedding layer
- Supports:
  - RNN
  - LSTM
  - GRU
- Uses multiple layers
- Applies dropout
- Outputs class logits

Your model must accept:
- model_type
- vocab_size
- embed_dim
- hidden_dim
- num_layers

Explain:
- The internal difference between RNN, LSTM, and GRU.


In [ ]:
class Model(nn.Module):
    def __init__(self, model_type, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.5, num_classes=4):
        super(Model, self).__init__()
        self.model_type = model_type.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        if self.model_type == 'rnn':
            self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        elif self.model_type == 'lstm':
            self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        elif self.model_type == 'gru':
            self.gru = nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        else:
            raise ValueError("model_type must be 'rnn', 'lstm', or 'gru'")
            
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        
        if self.model_type == 'rnn':
            output, hidden = self.rnn(embedded)
            last_hidden = hidden[-1]
        elif self.model_type == 'lstm':
            output, (hidden, cell) = self.lstm(embedded)
            last_hidden = hidden[-1]
        elif self.model_type == 'gru':
            output, hidden = self.gru(embedded)
            last_hidden = hidden[-1]
            
        last_hidden = self.dropout(last_hidden)
        out = self.fc(last_hidden)
        return out


### Explanation: Internal difference between RNN, LSTM, and GRU

- **Vanilla RNN**: Passes the hidden state from the previous step explicitly to the next unrolled cell by applying a simple non-linearity (like `tanh` or `ReLU`) to a weighted sum of the hidden state and current input. Prone to vanishing/exploding gradients since weights keep compounding over unrolled sequence length.
- **LSTM (Long Short-Term Memory)**: Uses an explicit internal memory structure called the *cell state* in addition to the hidden state, augmented by gates (input, forget, and output) formulated by sigmoids to protect information context over very long dependencies. 
- **GRU (Gated Recurrent Unit)**: Similar to LSTM with a gating mechanism, but does not possess a distinct separate cell state. It simplifies the gates into a *reset gate* and an *update gate*. It takes fewer parameters to compute than the LSTM, often computing faster but performing somewhat comparably.



# Part 4 – Training Loop

Implement:

- A full training loop
- Validation loop
- Accuracy computation
- Loss tracking per epoch

Train for 10-50 epochs.

Store:
- train_loss
- val_loss
- train_accuracy
- val_accuracy

Explain:
- Why do we use `model.train()` and `model.eval()`?


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total

def train(model, train_loader, val_loader, epochs=10, lr=0.001, verbose=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_accs = []
    val_accs = []
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
        train_loss, train_acc = evaluate(model, train_loader, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        
        if verbose:
            pass # print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
            
    return train_accs, val_accs, train_losses, val_losses


### Explanation: Why use `model.train()` and `model.eval()`?

1. **`model.train()`**: Tells PyTorch the model is currently in training mode. This ensures layers like `Dropout` or `BatchNorm` correctly operate stochastically on the batch (e.g., throwing away random connections for regularization).
2. **`model.eval()`**: Tells PyTorch to switch the model to evaluation or inference mode. Layers stop behaving stochastically (`Dropout` operates essentially as Identity, and `BatchNorm` uses running statistics rather than batch statistics), which provides deterministic responses during validation/testing.



# Part 5 – Model Comparison

Train:
- RNN
- LSTM
- GRU

Track validation accuracy and determine:
- Which performs best?
- Why?

Plot:
- Loss curves
- Accuracy curves

Explain signs of overfitting.


In [ ]:
# Train Models
vocab_dim = len(vocab)
embed_dimension = 64
hidden_dimension = 64
num_layers = 1

criterion = nn.BCEWithLogitsLoss() # or CrossEntropyLoss if doing multiclass


In [ ]:
num_classes = 4

print("Training RNN...")
rnn_model = Model(model_type='rnn', vocab_size=vocab_dim, embed_dim=embed_dimension, hidden_dim=hidden_dimension, num_layers=num_layers, num_classes=num_classes).to(device)
rnn_train_accs, rnn_val_accs, rnn_train_losses, rnn_val_losses = train(rnn_model, train_loader, val_loader, epochs=15, lr=0.001)

print("\nTraining LSTM...")
lstm_model = Model(model_type='lstm', vocab_size=vocab_dim, embed_dim=embed_dimension, hidden_dim=hidden_dimension, num_layers=num_layers, num_classes=num_classes).to(device)
lstm_train_accs, lstm_val_accs, lstm_train_losses, lstm_val_losses = train(lstm_model, train_loader, val_loader, epochs=15, lr=0.001)

print("\nTraining GRU...")
gru_model = Model(model_type='gru', vocab_size=vocab_dim, embed_dim=embed_dimension, hidden_dim=hidden_dimension, num_layers=num_layers, num_classes=num_classes).to(device)
gru_train_accs, gru_val_accs, gru_train_losses, gru_val_losses = train(gru_model, train_loader, val_loader, epochs=15, lr=0.001)


In [ ]:
# Plot Loss Curves
epochs_range = range(1, 16)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, rnn_val_losses, label='RNN Val Loss')
plt.plot(epochs_range, lstm_val_losses, label='LSTM Val Loss')
plt.plot(epochs_range, gru_val_losses, label='GRU Val Loss')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, rnn_val_accs, label='RNN Val Acc')
plt.plot(epochs_range, lstm_val_accs, label='LSTM Val Acc')
plt.plot(epochs_range, gru_val_accs, label='GRU Val Acc')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


### Explanation: Which performs best? Why?

The **LSTM** usually performs slightly better on textual sequences, closely followed by **GRU**.

**Why?**
The LSTM and GRU effectively combat the vanishing gradient problem using gating mechanisms (forget gate, input gate, update gate, reset gate). This allows them to capture long-range dependencies efficiently compared to vanilla RNNs, which suffer extensively from vanishing gradients where past contexts rapidly fade away. Consequently, they achieve lower validation losses faster and reach higher peak validation accuracy.

### Signs of Overfitting

When a neural network continues learning specific noise from the training data, its training loss will keep decreasing but its validation loss will stop decreasing and actually begin to rise, and validation accuracy degrades or plateaus. This increasing gap between training accuracy/loss and validation accuracy/loss is a classic sign of overfitting.



# Part 6 – Final Evaluation

Using the best model:

1. Evaluate on the test set.
2. Compute test accuracy.
3. Plot a confusion matrix.

Explain:
- Which classes are most confused?
- Why might that happen?


In [ ]:
# Evaluate the best model (LSTM) on test dataset
lstm_model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for text, label in test_loader:
        text, label = text.to(device), label.to(device)
        output = lstm_model(text).squeeze()
        preds = torch.argmax(output, dim=1)
        y_true.extend(label.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

test_acc = (y_pred == y_true).mean()
print(f"Final Test Accuracy: {test_acc:.4f}")


In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["World", "Sports", "Business", "Sci/Tech"])
fig, ax = plt.subplots(figsize=(6,6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title("Confusion Matrix for LSTM Model")
plt.show()


### Explanation: Which classes are most confused? Why might that happen?

Since the AG News dataset contains 4 different classes (World, Sports, Business, Sci/Tech) and they are coded numerically, confusion usually happens between closely related categories.
For example:
- **Business** and **Sci/Tech** are often confused because a technology company doing well (like Apple or Microsoft) may be reported as a business move or a tech innovation, containing vocabulary overlapping both.
- **World** and **Business** can intersect heavily during global economic events.

Models struggle when the distinguishing features rely on the context of the sentence rather than the individual pre-existing vocabulary (like just seeing "company" vs "software"). LSTMs handle sequential context better than RNNs, but when contexts overlap, errors happen.
